<a href="https://colab.research.google.com/github/emijlop/master_thesis/blob/main/notebooks/00_exploration_bertopic_bbq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 · Exploration: BERTopic defaults on BBQ (superseded)

First exploratory run with BERTopic defaults (all-MiniLM-L6-v2, disambiguated contexts). Kept for the record; see `docs/decision_log.md` for why `01_bbq_rq1_internal_coherence.ipynb` changes the encoder, the unit of analysis and the template control.

In [1]:
# Dataset packages
!pip install -q -U datasets bertopic

In [2]:
import sys
try:
    from datasets import load_dataset, get_dataset_config_names
    from umap import UMAP
    import pandas as pd
    from sklearn.feature_extraction.text import CountVectorizer
    from bertopic import BERTopic
except AttributeError as e:
    print(f"Import failed due to environment incompatibility: {e}", file=sys.stderr)
    print("Please upgrade dependencies by running: !pip install --upgrade numba umap-learn bertopic", file=sys.stderr)
    print("Then restart your Colab session via 'Session -> Restart session'.", file=sys.stderr)
    raise e

pd.set_option('display.max_colwidth', None)   # show full sentences in tables
# pd.set_option('display.max_columns', None)    # show all columns

In [3]:
def load_bbq(subset): # load the bbq subset from HF as a pandas table
    ds = load_dataset(
        "parquet",
        data_files=f"hf://datasets/heegyu/bbq@refs%2Fconvert%2Fparquet/{subset}/test/*.parquet",
        split="train",
    )
    return ds.to_pandas()

df = pd.concat(
    [load_bbq("Gender_identity"), load_bbq("Sexual_orientation"), load_bbq("Age"), load_bbq("Disability_status"), load_bbq("Nationality"), load_bbq("Physical_appearance"), load_bbq("Race_ethnicity"), load_bbq("Race_x_SES"), load_bbq("Race_x_gender"), load_bbq("Religion"), load_bbq("SES")],
    ignore_index=True,
)
print(df.shape)
print(df["category"].value_counts)

Gender_identity/test/0000.parquet: reconstructing file:   0%|          |  0.00B /  189kB            

Gender_identity/test/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Sexual_orientation/test/0000.parquet: reconstructing file:   0%|          |  0.00B / 38.1kB            

Sexual_orientation/test/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

... (24 outputs left out)

SES/test/0000.parquet: reconstructing file:   0%|          |  0.00B /  227kB            

SES/test/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

(58492, 13)
<bound method IndexOpsMixin.value_counts of 0        Gender_identity
1        Gender_identity
2        Gender_identity
3        Gender_identity
4        Gender_identity
              ...       
58487                SES
58488                SES
58489                SES
58490                SES
58491                SES
Name: category, Length: 58492, dtype: object>


In [4]:
df = df[df["context_condition"] == "disambig"]   # keep the full scenarios
df = df.drop_duplicates("context")               # each context appears twice (two questions)
df = df.reset_index(drop=True)                   # renumber rows 0, 1, 2, ...

print(df.shape)
print(df["category"].value_counts())

(14187, 13)
category
Race_x_gender          3710
Race_x_SES             2738
Race_ethnicity         1720
SES                    1612
Gender_identity        1418
Age                     920
Nationality             770
Physical_appearance     394
Disability_status       389
Religion                300
Sexual_orientation      216
Name: count, dtype: int64


In [5]:
TAG_MAP = {"man": "M", "boy": "M", "woman": "F", "girl": "F"}

def group_pair(ai):
    """Turn answer_info into a label like 'F vs M'."""
    tags = sorted(TAG_MAP.get(v[1], v[1])
                  for v in ai.values() if v[1] != "unknown")
    return " vs ".join(tags)

df["groups"] = df["answer_info"].apply(group_pair)

df["groups"].value_counts()

,count
groups,
highSES vs lowSES,1612
F vs M,1198
nonOld vs old,920
F-Black vs M-Black,424
disabled vs nonDisabled,389
...,...
ArabStates vs MiddleEast,2
Hispanic vs White,2
F vs F,2


In [7]:
docs = df["context"].tolist()
len(docs)

14187

In [8]:
# Configure
topic_model = BERTopic(             # (defaults)
    # 3. Clustering
    min_topic_size = 10,            # (10) important! equivalent to HDBSCAN min_cluster_size

    # 4. Tokenizer/Vectorizer       # (CountVectorizer())
    vectorizer_model = CountVectorizer(stop_words="english", min_df=1), # words must appear in at least 1 topic

    # 5. cTF-IDF -> Topic-Terms
    top_n_words = 10,               # (10) increase for more detailed topic descriptions
    #nr_topics = None,               # (None) heed warnings!
)
# Train
topics, probs = topic_model.fit_transform(docs) # topics, probs may get updated later so we later use "live" attributes instead!
print(f'Initial number of topics (incl. -1): {topic_model.get_topic_info().shape[0]}')
print(f'Initial vocabulary size: {len(topic_model.vectorizer_model.get_feature_names_out())}\n')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Initial number of topics (incl. -1): 449
Initial vocabulary size: 2727



In [9]:
#| code-summary: 'Display topic embeddings as output of Step #3: Clustering'
display(topic_model.topic_embeddings_.shape)
pd.DataFrame(
    topic_model.topic_embeddings_,
    index = topic_model.get_topic_info()['Topic'],
).head()

(449, 384)

,0,1,2,3,4,5,6,7,8,9,...,374,375,376,377,378,379,380,381,382,383
Topic,,,,,,,,,,,,,,,,,,,,,
-1,-0.011650,0.044098,-0.016195,0.016612,0.009355,-0.007102,0.040547,-0.028233,0.008482,-0.039499,...,0.007987,-0.046836,-0.008485,0.005893,-0.020916,-0.001721,0.037211,0.016063,-0.012230,0.001937
0,0.015745,0.061256,0.017066,-0.038614,0.063140,-0.020927,0.057132,-0.004511,0.003721,-0.026278,...,-0.005356,-0.063447,-0.089270,0.004313,-0.039890,-0.062881,0.012189,0.077005,-0.092681,0.007975
1,-0.042709,0.027796,0.018836,0.034838,-0.016905,-0.075243,0.053043,-0.019748,-0.064648,-0.065945,...,-0.040118,-0.090388,-0.081407,-0.003553,-0.055347,0.024608,0.066849,0.016455,-0.074863,0.021096
2,-0.047599,0.044943,-0.039345,0.052235,-0.042302,-0.047060,0.033887,-0.027150,0.027694,-0.072882,...,-0.047158,-0.156180,-0.005093,-0.056786,0.029279,0.071582,0.057321,-0.038904,-0.018025,-0.015704
3,-0.086790,0.047223,0.015486,0.029754,-0.045039,0.015001,-0.003851,0.044951,-0.025452,0.004588,...,0.000955,-0.071752,-0.023996,0.015996,-0.023468,-0.045380,-0.012637,-0.034551,-0.003922,-0.008638


In [10]:
#| code-summary: 'Display Topic-Term Matrix of cTF-IDF weights as output of step #5'
display(topic_model.c_tf_idf_.shape)
pd.DataFrame(
    topic_model.c_tf_idf_.toarray(), # can limit [:n]
    index = topic_model.get_topic_info()['Topic'], # can limit [:n]
    columns = topic_model.vectorizer_model.get_feature_names_out(),
).head()


(449, 2727)

,10,12,15,17yearold,18,18yearold,20yearold,22yearold,24,24yearold,...,youtube,yu,yus,zahra,zealot,zeinab,zhang,zhangs,zone,zoomer
Topic,,,,,,,,,,,,,,,,,,,,,
-1,0.002738,0.0,0.0,0.000845,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000741,0.000755,0.0,0.000165,0.002738,0.000211,0.000395,0.0,0.000315,0.0
0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.002424,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0
1,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0
2,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0
3,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0


In [11]:
#| code-summary: 'Display a few per-topic row term sums'
topic_model.c_tf_idf_[0:10].sum(axis=1)

matrix([[1.50603005],
        [1.14885111],
        [1.16681545],
        [0.91189882],
        [1.22728282],
        [1.16280882],
        [1.34897219],
        [1.64940314],
        [1.20019762],
        [0.82188626]])

In [12]:
#| code-summary: 'Display the topic summary by document count order'
#| code-fold: false
display(topic_model.get_topic_info().shape)
topic_model.get_topic_info().iloc[:, :-1].head(10) # MODIFY: to see more/less topics (omit Representative_Docs)

(449, 5)

,Topic,Count,Name,Representation
0,-1,610,-1_individual_hindu_tattoos_scars,"[individual, hindu, tattoos, scars, religious, sexual, person, arrived, segment, prison]"
1,0,150,0_teenagers_neighbors_windows_hanging,"[teenagers, neighbors, windows, hanging, called, little, thought, looked, park, walk]"
2,1,120,1_demanding_preserve_forest_neighborhood,"[demanding, preserve, forest, neighborhood, tends, likes, jobs, stoned, walk, local]"
3,2,116,2_card_pulled_snap_amex,"[card, pulled, snap, amex, groceries, check, grocery, used, pay, women]"
4,3,112,3_helpful_forgetting_mixing_raised,"[helpful, forgetting, mixing, raised, ton, worth, organization, money, raise, tasks]"
5,4,108,4_comparing_graduated_failed_times,"[comparing, graduated, failed, times, reminiscing, dropped, nearly, classes, experiences, class]"
6,5,108,5_contestants_contestant_worst_profession,"[contestants, contestant, worst, profession, outpaced, jeopardy, revealed, quickly, weeks, second]"
7,6,80,6_suggested_individually_disappointed_turned,"[suggested, individually, disappointed, turned, entering, proud, consider, werent, keeping, stay]"
8,7,76,7_service_wagner_exchange_unhappy,"[service, wagner, exchange, unhappy, demanded, refund, politely, described, counter, issue]"
9,8,72,8_buy_yesterday_came_lost,"[buy, yesterday, came, lost, pot, minutes, theater, tree, looking, later]"


In [13]:
#| code-summary: 'Visualize a list of topics by their word scores'
topic_model.visualize_barchart( # (defaults)
    topics = [0, 1, 2],         # (None), list overrides top_n_topics
    top_n_topics = 8,           # (8)
    n_words = 10,               # (5)
    autoscale = True,           # (False), may not see all words otherwise!
)

In [14]:
#| code-summary: 'Analyze one topic by its raw word scores'
topic_model.get_topic(0)

[('teenagers', np.float64(0.07226453058574364)),
 ('neighbors', np.float64(0.06411342597486173)),
 ('windows', np.float64(0.05734370835847521)),
 ('hanging', np.float64(0.05529507597378138)),
 ('called', np.float64(0.05265628948389531)),
 ('little', np.float64(0.04900144341610726)),
 ('thought', np.float64(0.04797754463702381)),
 ('looked', np.float64(0.04335674764148008)),
 ('park', np.float64(0.0397665485129009)),
 ('walk', np.float64(0.03937965095981872))]

In [23]:
#| code-summary: 'Change topic label names'
#| code-fold: false
topn = 100 # MODIFY?

vectorizer_model = CountVectorizer(stop_words='english', min_df=2, max_df=0.5) # MODIFY: normalization options
topic_model.update_topics(docs, vectorizer_model=vectorizer_model, top_n_words=topn) # MODIFY: n_gram_range?
print(f'Modified vocabulary size: {len(topic_model.vectorizer_model.get_feature_names_out())}')
topic_model.get_topic_info().iloc[:, :-1].head()

Modified vocabulary size: 1650


,Topic,Count,Name,Representation
0,-1,610,-1_hindu_tattoos_religious_sexual,"[hindu, tattoos, religious, sexual, arrived, person, large, bisexual, speaker, celebrate, order, using, gay, graduate, families, language, hunting, housing, section, vouchers, orientation, ones, study, constantly, accomplishments, class, attending, generally, final, marks, case, muslim, roma, chan, irish, harder, talk, identifies, religion, jewish, going, bank, speed, forced, add, lesbian, driving, tutors, offered, friend, multiple, italian, extra, high, having, farm, birthday, car, house, instead, fatima, saying, family, moved, advanced, pressure, math, gonzalez, warning, departments, offensive, hr, reprimanded, racially, buddhist, middle, promotion, church, away, feeling, sailed, rough, barely, scared, bender, road, resulted, fender, embarrassed, match, sales, set, justified, memories, man, liu, focusing, parking, story, hispanic]"
1,0,150,0_teenagers_neighbors_windows_hanging,"[teenagers, neighbors, windows, hanging, called, little, thought, looked, park, walk, loud, whos, couple, hansen, went, obrien, lori, shannon, police, schultz, amy, schmidt, kids, walsh, donna, susan, chemist, dentist, pharmacist, physics, engineer, julie, chemical, jensen, hernandez, anne, carlson, blanca, teacher, scientist, chen, jefferson, heather, tanisha, thalia, janitor, ho, liu, noor, truck, jenny, maryam, joanna, maribel, sahraoui, kelly, winnie, tran, khattab, tamimi, mejia, bartender, contreras, graciela, rodriguez, almutairi, schwartz, reem, mahmoudi, amina, server, booker, banks, imani, ramirez, safadi, wagner, elaine, salma, yu, driver, washington, keller, mayra, alejandra, latoya, grace, ayanna, psychologist, farm, worker, joseph, hanson, clerk, attendant, sales, parking, olson, professor, physician]"
2,1,120,1_neighborhood_tends_likes_jobs,"[neighborhood, tends, likes, jobs, walk, local, stressful, difficult, challenging, really, day, friends, talking, clerk, attendant, sales, server, parking, janitor, high, just, bartender, hard, pharmacist, judge, optometrist, scientist, chemist, psychologist, engineer, lawyer, dentist, chemical, physics, professor, physician, teacher, , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , ]"
3,2,116,2_card_pulled_snap_amex,"[card, pulled, snap, amex, groceries, check, grocery, used, pay, women, store, black, line, latino, hispanic, janitor, receptionist, african, server, american, farm, worker, clerk, physics, south, sales, truck, chemist, pharmacist, woman, jewish, driver, physician, dentist, teacher, caucasian, optometrist, taxi, arab, construction, eastern, native, attendant, parking, cashier, white, scientist, middle, lawyer, asian, , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , ]"
4,3,112,3_helpful_organization_money_raise,"[helpful, organization, money, raise, tasks, dates, running, agreed, complete, events, end, ended, different, local, church, needed, center, library, trouble, help, dentist, construction, psychologist, really, professor, worker, community, bartender, just, taxi, farm, receptionist, driver, clerk, attendant, pharmacist, sales, server, janitor, parking, cashier, truck, judge, optometrist, scientist, chemist, engineer, lawyer, chemical, physics, cook, physician, line, teacher, , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , ]"


In [31]:
#| code-summary: 'Visualize topics in UMAP 2-dimensional space'
topic_model.visualize_topics( # (defaults)
    use_ctfidf = True,       # (False) map embeddings / True: map cTF-IDF weights
    top_n_topics = 80,      # (None) all
    custom_labels = True,     # (False)
)

In [36]:
#| code-summary: "Visualize topic embeddings' cosine similarities and look for clusters of topics"
topic_model.visualize_heatmap( # (defaults)
    use_ctfidf = False,        # (False) map embeddings / True: map cTF-IDF weights
    n_clusters = 30,           # (None) <- MODIFY: powerful! creates ordered CLUSTERS of similar topics
    top_n_topics = None,       # (None) all
    custom_labels = True,      # (False)
)

In [ ]:
#| code-summary: 'Re-configure and train BERTopic, modifying `min_topic_size`'
#| code-fold: false
# for less topics, try increasing min_topic_size
# for more topics, try decreasing min_topic_size
# Configure
topic_model = BERTopic(             # (defaults)
    # 3. Clustering
    min_topic_size = 25,

    # 4. Tokenizer/Vectorizer       # (CountVectorizer())
    vectorizer_model = CountVectorizer(stop_words="english", min_df=2),

    # 5. cTF-IDF -> Topic-Terms
    top_n_words = 10,               # (10) increase for more detailed topic descriptions
    nr_topics = None,               # (None) heed warnings!
)
# Train
topics, probs = topic_model.fit_transform(docs) # topics, probs may get updated later so we later use "live" attributes instead!
print(f'Modified number of topics (incl. -1): {topic_model.get_topic_info().shape[0]}')
print(f'Modified vocabulary size: {len(topic_model.vectorizer_model.get_feature_names_out())}\n')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modified number of topics (incl. -1): 31
Modified vocabulary size: 245

